In [0]:
import logging as l
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
l.basicConfig(level=l.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = l.getLogger(__name__)

start_time = datetime.now()
logger.info("Gold layer started")


In [0]:

try:
    gold_main_df = spark.read.table("muse.silver.job_muse_silver_main")
    #complete table in gold for reference 
    gold_main_df.write.format("delta").mode("overwrite") \
        .option("path", "abfss://gold@musedatapipeline.dfs.core.windows.net/gold/main") \
        .saveAsTable("muse.gold.job_muse_gold_main")
    # Jobs by Company
    agg_company = gold_main_df.filter(F.col("company_name").isNotNull()) \
        .groupBy("company_name") \
        .agg(F.count("job_id").alias("job_count")) \
        .withColumn("_updated_at", F.current_timestamp())

    agg_company.write.format("delta").mode("overwrite") \
        .option("path", "abfss://gold@musedatapipeline.dfs.core.windows.net/gold/jobs_by_company") \
        .saveAsTable("muse.gold.jobs_by_company")

    # Jobs by Location
    agg_location = gold_main_df.filter(F.col("location-city").isNotNull()) \
        .groupBy("location-city", "location-state") \
        .agg(F.count("job_id").alias("job_count")) \
        .withColumn("_updated_at", F.current_timestamp())

    agg_location.write.format("delta").mode("overwrite") \
        .option("path", "abfss://gold@musedatapipeline.dfs.core.windows.net/gold/jobs_by_location") \
        .saveAsTable("muse.gold.jobs_by_location")

    # Top Hiring Companies
    # from pyspark.sql.window import Window

    window_spec = Window.orderBy(F.col("job_count").desc())

    agg_top_hiring = gold_main_df.filter(F.col("company_name").isNotNull()) \
        .groupBy("company_name") \
        .agg(F.count("job_id").alias("job_count")) \
        .withColumn("rank", F.dense_rank().over(window_spec)) \
        .withColumn("_updated_at", F.current_timestamp())

    agg_top_hiring.write.format("delta").mode("overwrite") \
        .option("path", "abfss://gold@musedatapipeline.dfs.core.windows.net/gold/top_hiring_companies") \
        .saveAsTable("muse.gold.top_hiring_companies")

    # Jobs by Experience Level
    agg_exp_level = gold_main_df.filter(F.col("exp_level").isNotNull()) \
        .groupBy("exp_level") \
        .agg(F.count("job_id").alias("job_count")) \
        .withColumn("_updated_at", F.current_timestamp())

    agg_exp_level.write.format("delta").mode("overwrite") \
        .option("path", "abfss://gold@musedatapipeline.dfs.core.windows.net/gold/jobs_by_exp_level") \
        .saveAsTable("muse.gold.jobs_by_exp_level")
    gold_main_df.show()
except Exception as e: 
    logger.error(f"Error: {e}")
    raise
finally:
    print("Gold layer done.")

In [0]:
%sql
select count(*) from muse.gold.job_muse_gold_main